In [10]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

def pose_to_homogeneous(pose):
    """
    pose: [x, y, z, Rx, Ry, Rz] (degrees)
    return: 4x4 homogeneous matrix
    """
    T = np.eye(4)
    T[:3, 3] = np.array(pose[:3]) / 1000.0  # mm -> m
    rot = R.from_euler('xyz', np.deg2rad(pose[3:]))
    T[:3, :3] = rot.as_matrix()
    return T

def invert_transform(T):
    R_inv = T[:3, :3].T
    t_inv = -R_inv @ T[:3, 3]
    T_inv = np.eye(4)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv
    return T_inv

def parse_pose_string(s):
    s = s.replace('[', '').replace(']', '').replace(',', '')
    numbers = [float(x) for x in s.split()]
    return numbers

# 경로 설정
csv_path = '/home/addinedu/lecture/Handeye_calibration_data.csv'

# 데이터 읽기
df = pd.read_csv(csv_path, sep='\t')

# 첫 번째 줄만 사용
row = df.iloc[0]

# 문자열 파싱
T_base_to_obj_pose = parse_pose_string(row['\uc815\ub2f5 \uc88c\ud45c 1'])
T_base_to_ee_pose = parse_pose_string(row['T_base_2_Endeffector(get_coords)'])
T_cam_to_obj_pose = parse_pose_string(row['T_camera_2_object (solvePnP + Rodrigues + xyztoEuler)'])

# 변환행렬로 변환
T_base_to_obj = pose_to_homogeneous(T_base_to_obj_pose)
T_base_to_ee = pose_to_homogeneous(T_base_to_ee_pose)
T_cam_to_obj = pose_to_homogeneous(T_cam_to_obj_pose)

# Hand-eye calibration (single)
T_ee_to_cam = np.linalg.inv(T_base_to_ee) @ T_base_to_obj @ np.linalg.inv(T_cam_to_obj)

print("T_EE_to_Camera (single object):")
print(T_ee_to_cam)

pose6d = homogeneous_to_pose(T_ee_to_cam)
print("6D pose (x, y, z, Rx, Ry, Rz):")
print(pose6d)


T_EE_to_Camera (single object):
[[ 0.64931689  0.76037173  0.0149134   0.02956285]
 [-0.75006004  0.64350577 -0.15267699 -0.01566768]
 [-0.12568812  0.08794981  0.98816361 -0.03945425]
 [ 0.          0.          0.          1.        ]]
6D pose (x, y, z, Rx, Ry, Rz):
[ 29.5628484  -15.66767697 -39.45424861   5.0861107    7.22049564
 -49.11770125]


In [14]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

def pose_to_homogeneous(pose):
    """
    [x, y, z, Rx, Ry, Rz] (degrees) -> 4x4 Homogeneous Transform
    """
    T = np.eye(4)
    T[:3, 3] = np.array(pose[:3]) / 1000.0  # mm → m
    rot = R.from_euler('xyz', np.deg2rad(pose[3:]))
    T[:3, :3] = rot.as_matrix()
    return T

def invert_transform(T):
    """
    Invert a 4x4 Homogeneous Transform
    """
    R_inv = T[:3, :3].T
    t_inv = -R_inv @ T[:3, 3]
    T_inv = np.eye(4)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv
    return T_inv

def parse_pose_string(s):
    """
    문자열을 받아서 6개 float 리스트로 변환
    """
    s = s.replace('[', '').replace(']', '').replace(',', '')
    numbers = [float(x) for x in s.split()]
    return numbers

def parse_two_poses_string(s):
    """
    문자열에서 물체 2개 pose를 분리해서 반환
    """
    parts = s.split('],')
    pose1 = parts[0].replace('[', '').replace(']', '').replace(',', '').split()
    pose2 = parts[1].replace('[', '').replace(']', '').replace(',', '').split()
    pose1 = [float(x) for x in pose1]
    pose2 = [float(x) for x in pose2]
    return pose1, pose2

def parse_pose_with_rotation_matrix(s):
    """
    SolvePnP 결과를 읽어 4x4 변환행렬로 변환
    문자열: '[tx, ty, tz], [R11, R12, R13, R21, R22, R23, R31, R32, R33]'
    """
    s = s.replace('[', '').replace(']', '').replace(',', '')
    numbers = [float(x) for x in s.split()]
    translation = np.array(numbers[0:3]) / 1000.0  # mm → m
    rotation_matrix = np.array(numbers[3:]).reshape(3, 3)

    T = np.eye(4)
    T[:3, 3] = translation
    T[:3, :3] = rotation_matrix
    return T

def homogeneous_to_pose(T, rot_type='euler', translation_in_mm=True):
    """
    4x4 변환행렬을 6D pose [x, y, z, Rx, Ry, Rz]로 변환
    """
    translation = T[:3, 3]
    if translation_in_mm:
        translation = translation * 1000.0  # m → mm

    rot = R.from_matrix(T[:3, :3])

    if rot_type == 'euler':
        rotation = np.rad2deg(rot.as_euler('xyz'))
    elif rot_type == 'rotvec':
        rotation = np.rad2deg(rot.as_rotvec())
    else:
        raise ValueError("rot_type must be 'euler' or 'rotvec'")

    pose6d = np.concatenate([translation, rotation])
    return pose6d

# ============================================================
# 메인 코드 시작
# ============================================================

# 경로 설정
csv_path = '/home/addinedu/lecture/Handeye_calibration_data_2.csv'

# 데이터 읽기
df = pd.read_csv(csv_path, sep='\t')

# 첫 번째 줄 사용
row = df.iloc[0]

# 문자열 파싱
T_base_to_obj_pose1, T_base_to_obj_pose2 = parse_two_poses_string(row['정답 좌표 2'])
T_base_to_ee_pose = parse_pose_string(row['T_base_2_Endeffector(get_coords)'])

# solvePnP로 얻은 T_camera_to_object는 "translation + rotation matrix" 형태로 저장돼 있음
T_cam_to_obj1 = parse_pose_with_rotation_matrix(row['T_camera_2_object (solvePnP + Rodrigues + xyztoEuler)'])
T_cam_to_obj2 = T_cam_to_obj1  # 두 번째도 동일하게 사용

# 변환행렬로 변환
T_base_to_obj1 = pose_to_homogeneous(T_base_to_obj_pose1)
T_base_to_obj2 = pose_to_homogeneous(T_base_to_obj_pose2)
T_base_to_ee = pose_to_homogeneous(T_base_to_ee_pose)

# 각각의 Hand-eye calibration 결과
T_ee_to_cam_1 = np.linalg.inv(T_base_to_ee) @ T_base_to_obj1 @ np.linalg.inv(T_cam_to_obj1)
T_ee_to_cam_2 = np.linalg.inv(T_base_to_ee) @ T_base_to_obj2 @ np.linalg.inv(T_cam_to_obj2)

# 최종 결과: 두 결과 평균
T_ee_to_cam = (T_ee_to_cam_1 + T_ee_to_cam_2) / 2

# 출력
print("=== T_EE_to_Camera (2개 물체 기반) ===")
print(T_ee_to_cam)

# 6D pose 변환
pose6d = homogeneous_to_pose(T_ee_to_cam)
print("\n=== T_EE_to_Camera (6D pose) [x(mm), y(mm), z(mm), Rx(deg), Ry(deg), Rz(deg)] ===")
print(pose6d)




=== T_EE_to_Camera (2개 물체 기반) ===
[[ 0.54917701 -0.00127429 -0.53108004  0.14752703]
 [ 0.5103782  -0.00166528 -0.50172028  0.07484854]
 [-0.04883462 -0.00477634  0.04836032  0.1564061 ]
 [ 0.          0.          0.          1.        ]]


ValueError: Non-positive determinant (left-handed or null coordinate frame) in rotation matrix 0: [[ 0.54917701 -0.00127429 -0.53108004]
 [ 0.5103782  -0.00166528 -0.50172028]
 [-0.04883462 -0.00477634  0.04836032]].

In [4]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

def pose_to_homogeneous(pose):
    """
    [x, y, z, Rx, Ry, Rz] (degrees) -> 4x4 Homogeneous Transform
    """
    T = np.eye(4)
    T[:3, 3] = np.array(pose[:3]) / 1000.0  # mm → m
    rot = R.from_euler('xyz', np.deg2rad(pose[3:]))
    T[:3, :3] = rot.as_matrix()
    return T

def invert_transform(T):
    """
    Invert a 4x4 Homogeneous Transform
    """
    R_inv = T[:3, :3].T
    t_inv = -R_inv @ T[:3, 3]
    T_inv = np.eye(4)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv
    return T_inv

def parse_two_poses_string(s):
    """
    [pose1], [pose2] 형태 문자열을 두 pose로 분리
    """
    parts = s.split('],')
    pose1 = parts[0].replace('[', '').replace(']', '').replace(',', '').split()
    pose2 = parts[1].replace('[', '').replace(']', '').replace(',', '').split()
    pose1 = [float(x) for x in pose1]
    pose2 = [float(x) for x in pose2]
    return pose1, pose2

def parse_pose_string(s):
    """
    [pose] 형태 문자열을 리스트로 변환
    """
    s = s.replace('[', '').replace(']', '').replace(',', '')
    numbers = [float(x) for x in s.split()]
    return numbers

def homogeneous_to_pose(T, rot_type='euler', translation_in_mm=True):
    """
    4x4 변환행렬을 6D pose [x, y, z, Rx, Ry, Rz]로 변환
    """
    translation = T[:3, 3]
    if translation_in_mm:
        translation = translation * 1000.0  # m → mm

    rot = R.from_matrix(T[:3, :3])

    if rot_type == 'euler':
        rotation = np.rad2deg(rot.as_euler('xyz'))
    elif rot_type == 'rotvec':
        rotation = np.rad2deg(rot.as_rotvec())
    else:
        raise ValueError("rot_type must be 'euler' or 'rotvec'")

    pose6d = np.concatenate([translation, rotation])
    return pose6d

def average_transforms(T_list):
    """
    여러 개의 4x4 변환행렬을 평균내기 (Rotation은 quaternion 평균, Translation은 그냥 평균)
    """
    translations = np.array([T[:3, 3] for T in T_list])
    mean_translation = np.mean(translations, axis=0)

    rotations = [R.from_matrix(T[:3, :3]) for T in T_list]
    quaternions = np.array([r.as_quat() for r in rotations])
    mean_quat = np.mean(quaternions, axis=0)
    mean_quat /= np.linalg.norm(mean_quat)  # normalize

    mean_rotation = R.from_quat(mean_quat).as_matrix()

    T_avg = np.eye(4)
    T_avg[:3, :3] = mean_rotation
    T_avg[:3, 3] = mean_translation
    return T_avg

# ============================================================
# 메인 코드 시작
# ============================================================

# 경로 설정
csv_path = '/home/addinedu/lecture/Handeye_calibration_data_2.csv'

# 데이터 읽기
df = pd.read_csv(csv_path, sep='\t')

# 첫 번째 줄 사용
row = df.iloc[0]

# 문자열 파싱
T_base_to_obj_pose1, T_base_to_obj_pose2 = parse_two_poses_string(row['정답 좌표 2'])
T_base_to_ee_pose = parse_pose_string(row['T_base_2_Endeffector(get_coords)'])
T_cam_to_obj_pose1, T_cam_to_obj_pose2 = parse_two_poses_string(row['T_camera_2_object (solvePnP + Rodrigues + xyztoEuler)'])

# 변환행렬로 변환
T_base_to_obj1 = pose_to_homogeneous(T_base_to_obj_pose1)
T_base_to_obj2 = pose_to_homogeneous(T_base_to_obj_pose2)
T_base_to_ee = pose_to_homogeneous(T_base_to_ee_pose)
T_cam_to_obj1 = pose_to_homogeneous(T_cam_to_obj_pose1)
T_cam_to_obj2 = pose_to_homogeneous(T_cam_to_obj_pose2)

# 각각 Hand-eye calibration 결과 계산
T_ee_to_cam_1 = np.linalg.inv(T_base_to_ee) @ T_base_to_obj1 @ np.linalg.inv(T_cam_to_obj1)
T_ee_to_cam_2 = np.linalg.inv(T_base_to_ee) @ T_base_to_obj2 @ np.linalg.inv(T_cam_to_obj2)

# Rotation은 quaternion 평균, Translation은 그냥 평균
T_ee_to_cam = average_transforms([T_ee_to_cam_1, T_ee_to_cam_2])


# 출력
print("=== T_EE_to_Camera_1 (2개 물체 기반) ===")
print(T_ee_to_cam_1)
# 출력
print("=== T_EE_to_Camera_2 (2개 물체 기반) ===")
print(T_ee_to_cam_2)
# 출력
print("=== T_EE_to_Camera (2개 물체 기반) ===")
print(T_ee_to_cam)

# 6D pose 변환
pose6d = homogeneous_to_pose(T_ee_to_cam)
print("\n=== T_EE_to_Camera (6D pose) [x(mm), y(mm), z(mm), Rx(deg), Ry(deg), Rz(deg)] ===")
print(pose6d)


=== T_EE_to_Camera_1 (2개 물체 기반) ===
[[ 0.63933775  0.74283963 -0.19858631  0.05478581]
 [-0.7618523   0.64691676 -0.03285993 -0.01439162]
 [ 0.10405915  0.17230203  0.97953239 -0.03869163]
 [ 0.          0.          0.          1.        ]]
=== T_EE_to_Camera_2 (2개 물체 기반) ===
[[ 0.63934945  0.75107222 -0.16468999  0.09341123]
 [-0.75702804  0.6523744   0.03627927 -0.02948007]
 [ 0.13468788  0.10147981  0.98567795 -0.03511632]
 [ 0.          0.          0.          1.        ]]
=== T_EE_to_Camera (2개 물체 기반) ===
[[ 0.63920836  0.74725743 -0.18171133  0.07409852]
 [-0.75970563  0.65026497  0.0016793  -0.02193584]
 [ 0.11941538  0.1369737   0.98335048 -0.03690398]
 [ 0.          0.          0.          1.        ]]

=== T_EE_to_Camera (6D pose) [x(mm), y(mm), z(mm), Rx(deg), Ry(deg), Rz(deg)] ===
[ 74.09852319 -21.9358437  -36.9039756    7.92986916  -6.85836383
 -49.92309484]


In [17]:
# Handeye 결과 각각 확인
print("det(T_ee_to_cam_1 rotation):", np.linalg.det(T_ee_to_cam_1[:3, :3]))
print("det(T_ee_to_cam_2 rotation):", np.linalg.det(T_ee_to_cam_2[:3, :3]))


det(T_ee_to_cam_1 rotation): -2.2213115173846814e-05
det(T_ee_to_cam_2 rotation): -2.2213115173846814e-05


In [38]:
calib = np.load("/home/addinedu/pymy_ws/eye_hand_result4.npz")
# camera_matrix = calib["camera_matrix"]
# dist_coeffs = calib["dist_coeffs"]
rot = calib["R"]
T_mat = calib["T"]
ttt = calib["t"]

pose_T_mat = homogeneous_to_pose(T_mat)

print("rot: ", rot)
print("trans: ", T_mat)
print("ttt: ", ttt)

print("mat: ", pose_T_mat)

print(calib)

rot:  [[ 0.6711663  -0.74080848  0.02717689]
 [ 0.74129714  0.67089363 -0.01950079]
 [-0.00378646  0.03323442  0.99944041]]
trans:  [[ 6.71166303e-01 -7.40808485e-01  2.71768932e-02  4.63298760e+01]
 [ 7.41297145e-01  6.70893630e-01 -1.95007850e-02 -2.64564430e+01]
 [-3.78645750e-03  3.32344231e-02  9.99440411e-01  1.61398849e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
ttt:  [[ 46.32987596]
 [-26.45644305]
 [ 16.13988485]]
mat:  [ 4.63298760e+04 -2.64564430e+04  1.61398849e+04  1.90455655e+00
  2.16948552e-01  4.78424836e+01]
NpzFile '/home/addinedu/pymy_ws/eye_hand_result4.npz' with keys: R, t, T


In [17]:
## 지혜님 코드 

import cv2
import numpy as np
# from ultralytics import YOLO
from scipy.spatial.transform import Rotation as R
import matplotlib.pyplot as plt

def homogeneous_to_pose(T, rot_type='euler', translation_in_mm=True):
    """
    4x4 변환행렬을 6D pose [x, y, z, Rx, Ry, Rz]로 변환
    """
    translation = T[:3, 3]
    if translation_in_mm:
        translation = translation * 1000.0  # m → mm

    rot = R.from_matrix(T[:3, :3])

    if rot_type == 'euler':
        rotation = np.rad2deg(rot.as_euler('xyz'))
    elif rot_type == 'rotvec':
        rotation = np.rad2deg(rot.as_rotvec())
    else:
        raise ValueError("rot_type must be 'euler' or 'rotvec'")

    pose6d = np.concatenate([translation, rotation])
    return pose6d

def show_detected_corners(img, corners, found, CHECKERBOARD, idx=None):
    """
    체커보드 인식 결과를 플로팅해 보여주는 함수
    """
    img_display = img.copy()
    
    if found:
        cv2.drawChessboardCorners(img_display, CHECKERBOARD, corners, found)
        title = f"Detected corners: img_{idx:02d}" if idx is not None else "Detected corners"
    else:
        title = f"Detection failed: img_{idx:02d}" if idx is not None else "Detection failed"
    
    img_display_rgb = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8,6))
    plt.imshow(img_display_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

calib = np.load("/home/addinedu/pymy_ws/calibration_refined4.npz")
mtx = calib["camera_matrix"]
dist = calib["dist_coeffs"]

print(mtx,"\n",dist)

CHECKERBOARD = (9, 6) 
SQUARE_SIZE = 20     # mm

objp = np.zeros((CHECKERBOARD[0]*CHECKERBOARD[1], 3), np.float32)
# opencv는 행열 (y,x) 로 찾음
objp[:, :2] = np.mgrid[0:CHECKERBOARD[1], 0:CHECKERBOARD[0]].T.reshape(-1, 2)
objp *= SQUARE_SIZE

R_target2cam = []
t_target2cam = []

M = np.array([
    [ 0, -1,  0],
    [-1,  0,  0],
    [ 0,  0,  1],
])


for i in range(22):
    img = cv2.imread(f"/home/addinedu/pymy_ws/img_{i:02}.jpg")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    found, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

    # 디텍션 결과 플로팅
    # show_detected_corners(img, corners, found, CHECKERBOARD, idx=i)

    if found:
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1),
                    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))

        ret, rvec, tvec = cv2.solvePnP(objp, corners2, mtx, dist)
        

        Ra, _ = cv2.Rodrigues(rvec)
        Ra_robot = M @ Ra @ M.T
        tvec_robot = M @ tvec
        R_target2cam.append(Ra_robot)
        t_target2cam.append(tvec_robot)
        print(f"체커보드 인식 성공: img_{i:02}.jpg")

    else:
        print(f"체커보드 인식 실패: img_{i:02}.jpg")


robot_coords_list = [[57.9, 83.4, 245.3, -82.25, -47.22, 3.84],[55.4, 54.6, 263.8, -80.57, -66.95, 2.59],
                    [36.9, 182.5, 293.8, -85.91, -44.86, 8.89],[94.3, 161.8, 234.6, -73.01, -56.85, -1.06],
                    [-20.7, 202.4, 253.7, -76.5, -57.0, -1.6],[21.8, 203.0, 225.2, -71.97, -55.99, -7.27],
                    [66.4, 4.3, 349.8, -89.06, -55.79, 12.31], [-72.9, 161.7, 226.8, -78.69, -27.19, -4.24],
                    [-73.4, 136.0, 314.5, -90.37, -42.31, 2.88], [-132.1, 137.7, 283.9, -85.33, -65.64, -15.63],
                    [-39.7, 120.2, 349.1, -99.2, -55.02, 10.96], [-62.7, 124.8, 382.0, -100.9, -46.48, 6.41],
                    [-118.3, 161.5, 331.4, -96.14, -48.66, 4.86], [-25.1, 190.8, 289.8, -80.89, -48.26, -0.48],
                    [-28.8, 197.2, 269.8, 11.78, -82.14, -96.55], [-39.8, 199.9, 229.0, -78.27, -25.26, -0.87],
                    [-31.3, 217.9, 168.9, -69.57, -52.21, -12.15], [-33.8, 221.9, 257.7, -77.66, -43.52, -3.81],
                    [-29.2, 216.5, 286.4, -80.78, -51.84, -3.9], [-11.0, 180.5, 346.6, -91.03, -38.66, 8.28],
                    [-133.0, 137.4, 319.1, -90.78, -40.11, -0.39], [93.3, 144.0, 280.8, -83.26, -51.67, 10.37]]      



R_gripper2base = []
t_gripper2base = []

for coords in robot_coords_list:
    x, y, z, rx, ry, rz = coords
    R_robot = R.from_euler('xyz', [rx, ry, rz], degrees=True).as_matrix()
    t_robot = np.array([[x], [y], [z]])
    
    # 여기서 바로 append하면 안됨! 먼저 역변환해야 함
    # T_base2gripper 생성
    T_base2gripper = np.eye(4)
    T_base2gripper[:3, :3] = R_robot
    T_base2gripper[:3, 3] = t_robot.flatten()

    # 역변환
    T_gripper2base = np.linalg.inv(T_base2gripper)

    R_gripper2base.append(T_gripper2base[:3, :3])
    t_gripper2base.append(T_gripper2base[:3, 3].reshape(3,1))

R_cam2ee, t_cam2ee = cv2.calibrateHandEye(
    R_gripper2base, t_gripper2base,
    R_target2cam, t_target2cam,
    method=cv2.CALIB_HAND_EYE_TSAI  # or CALIB_HAND_EYE_DANIILIDIS 가능??
)

# 정리
T_cam2ee = np.eye(4)
T_cam2ee[:3, :3] = R_cam2ee
T_cam2ee[:3, 3] = t_cam2ee.flatten()

np.savez("eye_hand_result6.npz", R=R_cam2ee, t=t_cam2ee, T=T_cam2ee)
print("저장굿")

print(homogeneous_to_pose(T_cam2ee))

[[970.42483372   0.         330.83954724]
 [  0.         967.08051293 173.53158829]
 [  0.           0.           1.        ]] 
 [[-0.44500398  0.57780108  0.00258172  0.00253966 -1.91404507]]
체커보드 인식 성공: img_00.jpg
체커보드 인식 성공: img_01.jpg
체커보드 인식 성공: img_02.jpg
체커보드 인식 성공: img_03.jpg
체커보드 인식 성공: img_04.jpg
체커보드 인식 성공: img_05.jpg
체커보드 인식 성공: img_06.jpg
체커보드 인식 성공: img_07.jpg
체커보드 인식 성공: img_08.jpg
체커보드 인식 성공: img_09.jpg
체커보드 인식 성공: img_10.jpg
체커보드 인식 성공: img_11.jpg
체커보드 인식 성공: img_12.jpg
체커보드 인식 성공: img_13.jpg
체커보드 인식 성공: img_14.jpg
체커보드 인식 성공: img_15.jpg
체커보드 인식 성공: img_16.jpg
체커보드 인식 성공: img_17.jpg
체커보드 인식 성공: img_18.jpg
체커보드 인식 성공: img_19.jpg
체커보드 인식 성공: img_20.jpg
체커보드 인식 성공: img_21.jpg
저장굿
[-8.92426498e+05  1.53496571e+06  1.27727185e+05  5.48366594e+01
  9.19289247e+00  2.72607692e+01]
